In [0]:
#Step 1 Look before you clean (cell 1) -- look at the data 
CATALOG = "capstone_supply_chain"

df = spark.table(f"{CATALOG}.bronze.raw_purchase_orders")
df.printSchema()
display(df)

In [0]:
#Build the Silver table (cell 2)
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

df_po = spark.table(f"{CATALOG}.bronze.raw_purchase_orders")

df_silver_po = (df_po
    # 1. Type casting
    .withColumn("order_qty", F.col("Order_Qty").cast("int"))
    .withColumn("unit_price", F.col("Unit_Price").cast("double"))
    .withColumn("order_date", F.to_date(F.col("Order_Date"), "yyyy-MM-dd"))
    .withColumn("expected_delivery", F.to_date(F.col("Expected_Delivery"), "yyyy-MM-dd"))
    # 2. Calculated fields
    .withColumn("line_total", F.col("order_qty") * F.col("unit_price"))
    .withColumn("lead_time_days", F.datediff(F.col("expected_delivery"), F.col("order_date")))
    # 3. Standardize status values
    .withColumn("status_clean",
        F.when(F.col("Status").ilike("%delivered%"), "Delivered")
         .when(F.col("Status").ilike("%transit%"), "In Transit")
         .when(F.col("Status").ilike("%pending%"), "Pending Approval")
         .when(F.col("Status").ilike("%confirmed%"), "Confirmed")
         .when(F.col("Status").ilike("%production%"), "In Production")
         .when(F.col("Status").ilike("%quality%"), "Quality Hold")
         .when(F.col("Status").ilike("%partial%"), "Partially Delivered")
         .otherwise(F.col("Status"))
    )
    # 4. Keep clean columns with snake_case names
    .select(
        F.col("PO_ID").alias("po_id"),
        F.col("Vendor_ID").alias("vendor_id"),
        F.col("Vendor_Name").alias("vendor_name"),
        F.col("Part_Number").alias("part_number"),
        F.col("Part_Description").alias("part_description"),
        "order_qty", "unit_price", "line_total",
        F.col("Currency").alias("currency"),
        "order_date", "expected_delivery", "lead_time_days",
        F.col("Plant_Code").alias("plant_code"),
        "status_clean",
        F.col("Notes").alias("notes"),
        F.col("_ingested_at").alias("bronze_ingested_at"),
        F.current_timestamp().alias("silver_processed_at"),
    )
    # 5. Data quality: drop rows with no PO ID, then remove duplicates
    .filter(F.col("po_id").isNotNull())
    .dropDuplicates(["po_id"])
)

df_silver_po.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.purchase_orders")
print("✅ silver.purchase_orders rows:", df_silver_po.count())
display(df_silver_po.limit(5))

In [0]:
#Quick quality check (cell 3)
df = spark.table(f"{CATALOG}.silver.purchase_orders")

# Are there nulls where we need real values?
df.select(
    F.sum(F.col("order_date").isNull().cast("int")).alias("null_order_date"),
    F.sum(F.col("line_total").isNull().cast("int")).alias("null_line_total"),
    F.sum(F.col("lead_time_days").isNull().cast("int")).alias("null_lead_time"),
).show()

# What status values do we have now?
df.groupBy("status_clean").count().orderBy("count", ascending=False).show()